In [129]:
import json, os
import pandas as pd

ROOT = r"D:\Compressed\archive_3"                       # adjust to your folder
META = f"{ROOT}/WLASL_v0.3.json"
VID_DIR = f"{ROOT}/videos"

with open(META) as f:
    meta = json.load(f)

rows = []
for entry in meta:                      # one entry per gloss
    for inst in entry["instances"]:     # one instance per video
        rows.append({
            "gloss": entry["gloss"],
            "video_id": inst["video_id"],
            "split": inst["split"],
            "signer_id": inst["signer_id"],
            "bbox": inst["bbox"],
            "frame_start": inst["frame_start"],
            "frame_end": inst["frame_end"],
            "path": f"{VID_DIR}/{inst['video_id']}.mp4",
        })

df = pd.DataFrame(rows)
df["exists"] = df["path"].apply(os.path.exists)
print(len(df), "listed,", df["exists"].sum(), "on disk")
df = df[df["exists"]]

21083 listed, 11980 on disk


In [130]:
df

,gloss,video_id,split,signer_id,bbox,frame_start,frame_end,path,exists
0,book,69241,train,118,"[385, 37, 885, 720]",1,-1,D:\Compressed\archive_3/videos/69241.mp4,True
10,book,07069,train,31,"[462, 44, 949, 720]",1,-1,D:\Compressed\archive_3/videos/07069.mp4,True
17,book,07068,train,36,"[234, 17, 524, 414]",1,-1,D:\Compressed\archive_3/videos/07068.mp4,True
22,book,07070,train,59,"[131, 26, 526, 480]",1,-1,D:\Compressed\archive_3/videos/07070.mp4,True
24,book,07099,val,12,"[162, 54, 528, 400]",1,-1,D:\Compressed\archive_3/videos/07099.mp4,True
...,...,...,...,...,...,...,...,...,...
21072,wheelchair,63047,train,11,"[39, 13, 248, 192]",1,-1,D:\Compressed\archive_3/videos/63047.mp4,True
21075,wheelchair,63050,train,12,"[163, 62, 625, 400]",1,-1,D:\Compressed\archive_3/videos/63050.mp4,True
21078,whistle,63186,train,2,"[76, 17, 236, 240]",1,-1,D:\Compressed\archive_3/videos/63186.mp4,True
21080,whistle,63188,train,11,"[68, 14, 212, 192]",1,-1,D:\Compressed\archive_3/videos/63188.mp4,True


In [131]:
top = df["gloss"].value_counts().head(100).index
df100 = df[df["gloss"].isin(top)]
print(df100.groupby(["gloss", "split"]).size().unstack(fill_value=0))

split        test  train  val
gloss                        
accident        2      9    2
apple           0      8    3
appointment     2      7    1
argue           1      8    1
bad             1      7    2
...           ...    ...  ...
woman           1      9    1
work            1      7    2
year            2      7    1
yes             1      9    2
yesterday       1      8    1

[100 rows x 3 columns]


In [132]:
import cv2

def probe(path):
    cap = cv2.VideoCapture(path)
    ok, frame = cap.read()
    info = dict(
        opened=cap.isOpened(),
        readable=ok,
        frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        fps=cap.get(cv2.CAP_PROP_FPS),
        w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    )
    cap.release()
    return info

probes = df100["path"].apply(probe).apply(pd.Series)
df100 = pd.concat([df100.reset_index(drop=True), probes.reset_index(drop=True)], axis=1)

bad = df100[~df100["readable"] | (df100["frames"] < 10)]
print(len(bad), "bad files")
df100 = df100.drop(bad.index)
df100.to_csv("clean_index.csv", index=False)
print(df100[["frames", "fps", "w", "h"]].describe())

0 bad files
            frames          fps            w            h
count  1120.000000  1120.000000  1120.000000  1120.000000
mean     64.668750    28.390436   828.616071   492.401786
std      25.835357     2.723300   586.908682   316.560536
min      19.000000    23.976024   288.000000   180.000000
25%      43.000000    25.000000   320.000000   240.000000
50%      63.000000    29.970000   656.000000   400.000000
75%      83.000000    29.970030  1280.000000   720.000000
max     195.000000    50.000000  1920.000000  1080.000000


In [133]:
# count videos per word per split
counts = df.groupby(["gloss", "split"]).size().unstack(fill_value=0)

# keep only words that have at least some train, val and test videos
good = counts[(counts["train"] >= 6) & (counts["val"] >= 1) & (counts["test"] >= 1)]
print(len(good), "words kept")

# keep only the videos for those words
df_sel = df[df["gloss"].isin(good.index)].reset_index(drop=True)
print(len(df_sel), "videos kept")
print(df_sel.groupby("split").size())

183 words kept
1801 videos kept
split
test      243
train    1272
val       286
dtype: int64


In [134]:
df_sel.gloss.nunique()

183

In [135]:
words = sorted(df_sel["gloss"].unique())
print(words)

['accident', 'add', 'africa', 'ago', 'alone', 'animal', 'appointment', 'argue', 'australia', 'baby', 'bad', 'balance', 'ball', 'balloon', 'banana', 'bar', 'barely', 'basketball', 'because', 'before', 'big', 'bird', 'blanket', 'blue', 'bowling', 'bring', 'brother', 'brown', 'buy', 'call', 'can', 'candy', 'carrot', 'cat', 'catch', 'champion', 'change', 'cheat', 'check', 'child', 'choose', 'city', 'close', 'cloud', 'cold', 'color', 'computer', 'convince', 'cool', 'correct', 'country', 'cousin', 'crash', 'cry', 'dark', 'day', 'deaf', 'delay', 'delicious', 'dive', 'doctor', 'dog', 'drink', 'environment', 'example', 'far', 'farm', 'fast', 'fat', 'fine', 'finish', 'fish', 'follow', 'full', 'future', 'girl', 'give', 'glasses', 'go', 'good', 'government', 'grammar', 'great', 'group', 'happy', 'headache', 'help', 'hope', 'how', 'humble', 'important', 'inform', 'interest', 'jealous', 'kiss', 'language', 'last', 'laugh', 'leave', 'letter', 'like', 'list', 'lose', 'make', 'match', 'meet', 'minute',

In [136]:
sizes = df_sel.groupby("gloss").size().sort_values(ascending=False)
print(sizes)

gloss
before       16
cool         16
thin         16
drink        15
go           15
             ..
test          8
underwear     8
when          8
watch         8
weather       8
Length: 183, dtype: int64


In [137]:
my_words = [
    "yes", "no", "help", "good", "bad", "want", "like", "go", "stay", "wait", "more", "finish",
    "give", "take", "call", "meet", "work", "watch", "tell", "buy", "drink",
    "what", "who", "when", "how",
    "happy", "sad", "sick", "scared", "cold",
    "doctor", "girl", "woman", "baby",
    "today", "yesterday", "soon",
    "office", "problem", "computer",
]
df_final = df_sel[df_sel["gloss"].isin(my_words)].reset_index(drop=True)
print(df_final.groupby("split").size())
print(df_final["gloss"].nunique(), "words,", len(df_final), "videos")

split
test      54
train    286
val       69
dtype: int64
40 words, 409 videos


In [138]:
df_final

,gloss,video_id,split,signer_id,bbox,frame_start,frame_end,path,exists
0,drink,69302,val,115,"[551, 68, 1350, 1080]",1,-1,D:\Compressed\archive_3/videos/69302.mp4,True
1,drink,65539,train,94,"[153, 11, 488, 370]",1,-1,D:\Compressed\archive_3/videos/65539.mp4,True
2,drink,17710,train,36,"[196, 15, 521, 414]",1,-1,D:\Compressed\archive_3/videos/17710.mp4,True
3,drink,17733,train,12,"[186, 63, 551, 400]",1,-1,D:\Compressed\archive_3/videos/17733.mp4,True
4,drink,65540,train,94,"[167, 19, 480, 370]",1,-1,D:\Compressed\archive_3/videos/65540.mp4,True
...,...,...,...,...,...,...,...,...,...
404,watch,62444,train,13,"[132, 0, 541, 480]",1,-1,D:\Compressed\archive_3/videos/62444.mp4,True
405,watch,62445,train,13,"[125, 0, 526, 480]",1,-1,D:\Compressed\archive_3/videos/62445.mp4,True
406,watch,62439,train,38,"[436, 55, 831, 720]",1,-1,D:\Compressed\archive_3/videos/62439.mp4,True
407,watch,62452,train,11,"[78, 18, 218, 192]",1,-1,D:\Compressed\archive_3/videos/62452.mp4,True


In [139]:
from extracted_keypoints import *
download_models()
hand, pose = make_landmarkers()

row = df_final.iloc[0]
seq, rate = extract_video(row, hand, pose)
print(row["gloss"], seq.shape, rate)

drink (32, 225) 0.6875


In [140]:
debug_draw(row, hand, pose)

'debug.jpg'

In [141]:
from custom_rec import load_index, assign_splits

custom = assign_splits(load_index(), val_per_word=2, test_per_word=2)
custom = run_all(custom, hand, pose)
custom.to_csv("custom_with_rates.csv", index=False)   # so you never re-extract
print(custom["hand_rate"].describe())

count    240.000000
mean       0.662630
std        0.286118
min        0.062500
25%        0.375000
50%        0.718750
75%        0.937500
max        1.000000
Name: hand_rate, dtype: float64


In [142]:
df_done = run_all(df_final, hand, pose)
df_done.to_csv("index_with_rates.csv", index=False)
print(df_done["hand_rate"].describe())

count    409.000000
mean       0.681158
std        0.225878
min        0.218750
25%        0.500000
50%        0.656250
75%        1.000000
max        1.000000
Name: hand_rate, dtype: float64


In [143]:
import pandas as pd

wlasl = pd.read_csv("index_with_rates.csv", dtype={"video_id": str})
wlasl_clean = wlasl[(wlasl["hand_rate"] >= 0.3) & (wlasl["gloss"] != "woman")]

custom = pd.read_csv("custom_with_rates.csv", dtype={"video_id": str})
custom_clean = custom[custom["hand_rate"] >= 0.15]

df_clean = pd.concat([wlasl_clean, custom_clean], ignore_index=True)
df_clean.to_csv("df_clean_combined.csv", index=False)   # save it so you don't rebuild this every restart

print(df_clean.groupby(["split"]).size())
print(df_clean.groupby(["gloss", "split"]).size().unstack(fill_value=0).head())

split
test      77
train    462
val       94
dtype: int64
split  test  train  val
gloss                  
I         2     26    2
baby      1      6    1
bad       1      7    2
buy       1      6    2
call      4     24    4


In [144]:
wlasl = df_done[df_done["hand_rate"] >= 0.3].copy()
wlasl["source"] = "wlasl"

mine = custom[custom["hand_rate"] >= 0.3].copy()
mine["source"] = "custom"
mine = mine[mine["gloss"].isin(wlasl["gloss"].unique())]   # see note below

df_clean = pd.concat([wlasl, mine], ignore_index=True)

counts = df_clean.groupby(["gloss", "split"]).size().unstack(fill_value=0)
print(counts[(counts == 0).any(axis=1)])    # words missing a split
print(df_clean.groupby(["source", "split"]).size())

split  test  train  val
gloss                  
woman     0      9    1
source  split
custom  test      16
        train     98
        val       18
wlasl   test      52
        train    284
        val       69
dtype: int64


CHECKING NUMBER OF VIDEOS IN A GLOSS

In [145]:
len(df_clean[df_clean["gloss"] == "mother"])

0

In [146]:
print(df_clean["gloss"].nunique())
print(sorted(df_clean["gloss"].unique()))

40
['baby', 'bad', 'buy', 'call', 'cold', 'computer', 'doctor', 'drink', 'finish', 'girl', 'give', 'go', 'good', 'happy', 'help', 'how', 'like', 'meet', 'more', 'no', 'office', 'problem', 'sad', 'scared', 'sick', 'soon', 'stay', 'take', 'tell', 'today', 'wait', 'want', 'watch', 'what', 'when', 'who', 'woman', 'work', 'yes', 'yesterday']


In [147]:
df_clean = df_clean[df_clean["gloss"] != "woman"].reset_index(drop=True)

labels = sorted(df_clean["gloss"].unique())
label_to_idx = {w: i for i, w in enumerate(labels)}
df_clean["label"] = df_clean["gloss"].map(label_to_idx)

import json
json.dump(labels, open("labels.json", "w"))

print(len(labels), "words,", len(df_clean), "videos")
print(df_clean.groupby("split").size())
print(labels[:15])

39 words, 527 videos
split
test      68
train    373
val       86
dtype: int64
['baby', 'bad', 'buy', 'call', 'cold', 'computer', 'doctor', 'drink', 'finish', 'girl', 'give', 'go', 'good', 'happy', 'help']


In [148]:
def normalize(seq):                       # seq: one video, shape (32, 225)
    p = seq.reshape(len(seq), 75, 3).copy()   # 75 points, 3 numbers each
    for t in range(len(p)):
        l, r = p[t, 11], p[t, 12]         # left and right shoulder
        if not (l.any() and r.any()):
            continue                      # shoulders not found, leave this frame alone
        center = (l + r) / 2
        scale = np.linalg.norm(l[:2] - r[:2]) + 1e-6
        found = p[t].any(axis=1)          # only points that were actually detected
        p[t, found] = (p[t, found] - center) / scale
    return p.reshape(len(p), 225)


In [149]:
import numpy as np

def load_split(name):
    d = df_clean[df_clean["split"] == name]
    X = np.stack([np.load(f"data/keypoints/{v}.npy") for v in d["video_id"]])
    y = d["label"].values
    return X, y

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

X_train_n = np.stack([normalize(s) for s in X_train])
X_val_n = np.stack([normalize(s) for s in X_val])
X_test_n = np.stack([normalize(s) for s in X_test])

np.savez("data_normalized.npz", X_train=X_train_n, y_train=y_train,
         X_val=X_val_n, y_val=y_val, X_test=X_test_n, y_test=y_test)
print(X_train_n.shape, X_val_n.shape, X_test_n.shape)

(373, 32, 225) (86, 32, 225) (68, 32, 225)


In [150]:
import numpy as np, json
import torch, torch.nn as nn

data = np.load("data_normalized.npz")
Xtr = torch.tensor(data["X_train"], dtype=torch.float32)
ytr = torch.tensor(data["y_train"], dtype=torch.long)
Xva = torch.tensor(data["X_val"], dtype=torch.float32)
yva = torch.tensor(data["y_val"], dtype=torch.long)
Xte = torch.tensor(data["X_test"], dtype=torch.float32)
yte = torch.tensor(data["y_test"], dtype=torch.long)

labels = json.load(open("labels.json"))

In [151]:
class SignGRU(nn.Module):
    def __init__(self, n_classes, in_dim=225, hidden=128):
        super().__init__()
        self.gru = nn.GRU(in_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x):
        out, h = self.gru(x)
        return self.fc(self.drop(h[-1]))

model = SignGRU(len(labels))
print(model(Xtr[:4]).shape)

torch.Size([4, 39])


In [152]:
def augment(xb):
    B, T, _ = xb.shape
    p = xb.reshape(B, T, 75, 3).clone()
    present = (p != 0).any(-1, keepdim=True).float()   # which points were actually detected

    # random tilt in the x-y plane
    a = torch.empty(B, 1, 1).uniform_(-0.15, 0.15)
    x, y = p[..., 0].clone(), p[..., 1].clone()
    p[..., 0] = torch.cos(a) * x - torch.sin(a) * y
    p[..., 1] = torch.sin(a) * x + torch.cos(a) * y

    # random size change
    p = p * torch.empty(B, 1, 1, 1).uniform_(0.9, 1.1)

    # tiny noise, only on detected points
    p = p + torch.randn_like(p) * 0.01 * present

    # random trim of a few frames at the start and end, then resample back to 32
    out = torch.empty_like(p)
    for b in range(B):
        st = np.random.randint(0, 4)
        en = T - 1 - np.random.randint(0, 4)
        idx = torch.linspace(st, en, T).round().long()
        out[b] = p[b, idx]
    return out.reshape(B, T, 225)

print(augment(Xtr[:4]).shape)    # should print torch.Size([4, 32, 225])

torch.Size([4, 32, 225])


In [153]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(1)
    return (pred == y).float().mean().item()

best_val, best_state, wait = 0, None, 0

model = SignGRU(len(labels))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
best_val, best_state, wait = 0, None, 0

for epoch in range(250):
    model.train()
    total = 0
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = loss_fn(model(augment(xb)), yb)      # augmented copy of the batch
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)

    tr_acc = accuracy(Xtr, ytr)
    va_acc = accuracy(Xva, yva)
    if epoch % 5 == 0:
        print(f"epoch {epoch}: loss {total/len(Xtr):.3f}  train acc {tr_acc:.2f}  val acc {va_acc:.2f}")

    if va_acc > best_val:
        best_val, wait = va_acc, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_state, "best_model_aug.pt")
    else:
        wait += 1
        if wait >= 30:
            print("no improvement for 30 epochs, stopping")
            break

model.load_state_dict(best_state)
print("best val accuracy:", round(best_val, 3))

epoch 0: loss 3.637  train acc 0.15  val acc 0.06
epoch 5: loss 2.849  train acc 0.31  val acc 0.17
epoch 10: loss 2.194  train acc 0.50  val acc 0.28
epoch 15: loss 1.766  train acc 0.59  val acc 0.24
epoch 20: loss 1.432  train acc 0.70  val acc 0.33
epoch 25: loss 1.132  train acc 0.73  val acc 0.34
epoch 30: loss 0.990  train acc 0.82  val acc 0.45
epoch 35: loss 0.837  train acc 0.89  val acc 0.34
epoch 40: loss 0.611  train acc 0.92  val acc 0.37
epoch 45: loss 0.468  train acc 0.94  val acc 0.38
epoch 50: loss 0.437  train acc 0.92  val acc 0.43
epoch 55: loss 0.449  train acc 0.91  val acc 0.35
epoch 60: loss 0.277  train acc 0.98  val acc 0.45
epoch 65: loss 0.196  train acc 0.99  val acc 0.44
epoch 70: loss 0.150  train acc 0.97  val acc 0.35
epoch 75: loss 0.209  train acc 0.97  val acc 0.42
epoch 80: loss 0.313  train acc 0.95  val acc 0.40
epoch 85: loss 0.102  train acc 1.00  val acc 0.40
no improvement for 30 epochs, stopping
best val accuracy: 0.477


In [154]:
before = len(df_done[df_done["hand_rate"] >= 0.3])
after = len(df_clean)
print(f"wlasl only: {before}")
print(f"wlasl + custom: {after}")
print(f"custom rows added: {after - before}")

wlasl only: 405
wlasl + custom: 527
custom rows added: 122


In [155]:
m1 = SignGRU(len(labels)); m1.load_state_dict(torch.load("best_model_baseline.pt")); m1.eval()
m2 = SignGRU(len(labels)); m2.load_state_dict(torch.load("best_model_aug.pt")); m2.eval()
with torch.no_grad():
    p1 = m1(torch.tensor(X_test_n, dtype=torch.float32)).argmax(1)
    p2 = m2(torch.tensor(X_test_n, dtype=torch.float32)).argmax(1)
print("predictions identical:", (p1 == p2).all().item())
print("agreement rate:", (p1 == p2).float().mean().item())

RuntimeError: Error(s) in loading state_dict for SignGRU:
	size mismatch for fc.weight: copying a param with shape torch.Size([41, 128]) from checkpoint, the shape in current model is torch.Size([39, 128]).
	size mismatch for fc.bias: copying a param with shape torch.Size([41]) from checkpoint, the shape in current model is torch.Size([39]).

In [ ]:
m1 = SignGRU(len(labels_baseline)); m1.load_state_dict(torch.load("best_model_baseline.pt")); m1.eval()
m2 = SignGRU(len(labels_custom)); m2.load_state_dict(torch.load("best_model_aug.pt")); m2.eval()

with torch.no_grad():
    p1 = m1(torch.tensor(X_test_n, dtype=torch.float32)).argmax(1)
    p2 = m2(torch.tensor(X_test_n, dtype=torch.float32)).argmax(1)

print("predictions identical:", (p1 == p2).all().item())
print("agreement rate:", (p1 == p2).float().mean().item())

def load_and_eval(state_path, X, y, labels):
    m = SignGRU(len(labels))
    m.load_state_dict(torch.load(state_path))
    m.eval()
    with torch.no_grad():
        pred = m(torch.tensor(X, dtype=torch.float32)).argmax(1)
    return (pred == torch.tensor(y)).float().mean().item()

print("baseline test acc:", load_and_eval("best_model_baseline.pt", X_test_n, y_test, labels_baseline))
print("custom-added test acc:", load_and_eval("best_model_aug.pt", X_test_n, y_test, labels_custom))

RuntimeError: Error(s) in loading state_dict for SignGRU:
	size mismatch for fc.weight: copying a param with shape torch.Size([41, 128]) from checkpoint, the shape in current model is torch.Size([39, 128]).
	size mismatch for fc.bias: copying a param with shape torch.Size([41]) from checkpoint, the shape in current model is torch.Size([39]).

In [ ]:
m1 = SignGRU(len(labels_baseline)); m1.load_state_dict(torch.load("best_model_baseline.pt")); m1.eval()
m2 = SignGRU(len(labels_custom)); m2.load_state_dict(torch.load("best_model_aug.pt")); m2.eval()

SignGRU(
  (gru): GRU(225, 128, batch_first=True)
  (drop): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=128, out_features=39, bias=True)
)

In [ ]:
labels = sorted(df_clean["gloss"].unique())
json.dump(labels, open("labels.json", "w"))

In [ ]:
import json
labels = json.load(open("labels.json"))
print(len(labels))
print(labels)

41
['baby', 'bad', 'buy', 'call', 'cold', 'computer', 'doctor', 'drink', 'finish', 'girl', 'give', 'go', 'good', 'happy', 'help', 'how', 'like', 'meet', 'more', 'mother', 'no', 'office', 'problem', 'sad', 'scared', 'sick', 'soon', 'stay', 'sweet', 'take', 'tell', 'today', 'wait', 'want', 'watch', 'what', 'when', 'who', 'work', 'yes', 'yesterday']


In [ ]:
import pandas as pd
df_check = pd.read_csv("df_clean_combined.csv", dtype={"video_id": str})   # or whatever your current merged file is called
print(df_check[df_check["gloss"] == "woman"].groupby("split").size())

Series([], dtype: int64)


In [ ]:
import pandas as pd, json

df_clean = pd.read_csv("df_clean_combined.csv", dtype={"video_id": str})

# sanity check: does every word have train, val AND test coverage?
counts = df_clean.groupby(["gloss", "split"]).size().unstack(fill_value=0)
missing = counts[(counts == 0).any(axis=1)]
print("words missing a split:", list(missing.index))
print(counts.min())

words missing a split: []
split
test     1
train    4
val      1
dtype: int64


In [ ]:
import json

labels = sorted(df_clean["gloss"].unique())
label_to_idx = {w: i for i, w in enumerate(labels)}
df_clean["label"] = df_clean["gloss"].map(label_to_idx)
json.dump(labels, open("labels.json", "w"))

print(len(labels), labels)
print(df_clean.groupby("split").size())

41 ['baby', 'bad', 'buy', 'call', 'cold', 'computer', 'doctor', 'drink', 'finish', 'girl', 'give', 'go', 'good', 'happy', 'help', 'how', 'like', 'meet', 'more', 'mother', 'no', 'office', 'problem', 'sad', 'scared', 'sick', 'soon', 'stay', 'sweet', 'take', 'tell', 'today', 'wait', 'want', 'watch', 'what', 'when', 'who', 'work', 'yes', 'yesterday']
split
test      67
train    315
val       86
dtype: int64


Checking number of classes on thte latest model

In [ ]:
import torch
ckpt = torch.load("best_model_aug.pt", map_location="cpu")
print(type(ckpt))          
print(list(ckpt.keys())[:10])   
print(ckpt["fc.weight"].shape[0], "classes")

<class 'dict'>
['gru.weight_ih_l0', 'gru.weight_hh_l0', 'gru.bias_ih_l0', 'gru.bias_hh_l0', 'fc.weight', 'fc.bias']
43 classes
